# 15 · One agent, built from nothing

The board can now tell you that a number moved. Somebody still has to find out
why, and that somebody is asleep.

This notebook builds **one** agent, in cells, and gets it to do a real piece of
work against the real warehouse. Notebook 16 turns it into five.

No experience with agents needed. We build up one word at a time.

In [ ]:
import sys; sys.path.insert(0, '..')
import os
from pipelines.lib.config import dsn, SCHEMA      # loads .env, so the key is set

print('model :', os.environ.get('ONCALL_MODEL', 'openai:gpt-5.4-mini'))
print('key   :', 'present' if os.environ.get('OPENAI_API_KEY') else 'MISSING')

---

## Word 1 · a **tool**

A tool is a python function the model is allowed to call.

That is the whole idea. The model cannot query your database; it can ask you to,
by name, with arguments, and read what you hand back.

In [ ]:
from langchain.tools import tool

@tool
def count_rows(table: str) -> str:
    """Count the rows in one warehouse table."""
    import psycopg
    with psycopg.connect(dsn()) as c:
        c.read_only = True
        n = c.execute(f'SELECT count(*) FROM {SCHEMA}.{table}').fetchone()[0]
    return f'{table} has {n:,} rows'

print('name        :', count_rows.name)
print('description :', count_rows.description)
print('arguments   :', count_rows.args)
print()
print(count_rows.invoke({'table': 'bronze_driver_app'}))

### The docstring is not documentation, it is the interface

The model chooses tools by reading that description. A vague docstring produces
an agent that picks the wrong tool and then explains confidently why it was
right.

Notice `c.read_only = True`. Hold that thought.

---

## Word 2 · an **agent**

A model, some tools, and a loop: call the model, run any tool it asked for, give
it the result, repeat until it stops asking.

`create_agent` is that loop.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[count_rows],
    system_prompt='You answer questions about a data warehouse using the tools you have.',
)

result = agent.invoke({'messages': [
    {'role': 'user', 'content': 'How many rows are in bronze_driver_app and silver_rides?'}
]})

for m in result['messages']:
    kind = type(m).__name__.replace('Message', '')
    if getattr(m, 'tool_calls', None):
        for call in m.tool_calls:
            print(f'  {kind:10} calls {call["name"]}({call["args"]})')
    elif m.content:
        print(f'  {kind:10} {str(m.content)[:150]}')

**Read the trace.** The model decided to call the tool twice, we ran it twice,
and it wrote the answer from what came back. Nobody wrote an if statement.

---

## Word 3 · **structured output**

Prose is readable and unusable. If another program has to act on the answer, it
needs fields, not a paragraph.

In [ ]:
from pydantic import BaseModel, Field

class TableVerdict(BaseModel):
    table: str
    rows: int
    looks_empty: bool = Field(description="true if the table has no rows")
    comment: str

typed = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[count_rows],
    system_prompt='You answer questions about a data warehouse.',
    response_format=TableVerdict,
)

out = typed.invoke({'messages': [
    {'role': 'user', 'content': 'Check bronze_driver_app'}]})

v = out['structured_response']
print(type(v).__name__, '\n')
print('  table      ', v.table)
print('  rows       ', v.rows)
print('  looks_empty', v.looks_empty, '  <- a bool you can branch on')
print('  comment    ', v.comment)

`v.looks_empty` is a real python boolean. **That is what lets a supervisor stop
early**, and it is why every agent in this project returns a schema.

---

## Word 4 · the **guard**

The agent above could have been asked to run any SQL. It was not, because the
tool only counts rows. But the moment you give it a general query tool, the
question becomes: what stops it writing?

In [ ]:
from agent_service.tools.warehouse import run_sql

print(run_sql.invoke({'query': 'SELECT count(*) FROM teach.silver_rides'}))
print()
print(run_sql.invoke({'query': 'DELETE FROM teach.silver_rides'}))
print()
print(run_sql.invoke({'query': 'SELECT 1; DROP TABLE teach.gold_daily'}))

### Two guards, and only one of them is about trust

**The syntactic guard** rejects anything that is not a single SELECT. Easy to
read, easy to explain, and easy to fool: it is a string check.

**The read only transaction** is the one that matters:

```python
conn.read_only = True
```

Postgres itself refuses the write, no matter what the model asked for or how the
query was spelled. Say the distinction out loud, because it generalises:

> **Whether an agent meant well is a judgement. Whether it CAN write is a fact.**

Build on facts. Every rule you enforce only in a system prompt is a rule you are
hoping about.

---

## Now a real one: the data detective

Same three ideas, pointed at a real breach.

In [ ]:
from agent_service.tools.warehouse import READ_TOOLS

for t in READ_TOOLS:
    print(f'  {t.name:20} {t.description.splitlines()[0][:70]}')

In [ ]:
from agent_service.agents import data_agent, DATA_PROMPT

print(DATA_PROMPT)

### Read that prompt again

It contains a **method**, not answers. Look at the warehouse, in this order,
and quote real numbers. Nowhere does it say "if surge is missing, look at app
versions".

That matters more than anything else in this notebook. An agent given a list of
known problems handles known problems. An agent given a method handles the one
nobody has seen.

## Run it against something that is actually broken

In [ ]:
from signal_service import evaluate as ev
from signal_service.kpis import get

kpi = get('surge_coverage_pct')
reading, verdict = ev.evaluate(kpi)
breach = ev.to_breach(kpi, reading, verdict)
print(breach.one_line())
print()
print('breached:', verdict.breached)
if not verdict.breached:
    print('\nNothing is broken. Break it first, in a terminal:')
    print('   python break_it.py && python cli.py run all')

In [ ]:
agent = data_agent()

out = agent.invoke({'messages': [{'role': 'user', 'content':
    f'{breach.one_line()}\n\nThe KPI means: {breach.means}\n\n'
    f'What is wrong with the data?'}]})

calls = [c['name'] for m in out['messages'] for c in (getattr(m, 'tool_calls', None) or [])]
print('tools it chose to use:', calls, '\n')

v = out['structured_response']
print('FINDING  ', v.finding)
print('\nEVIDENCE\n', v.evidence)
print('\nSCALE    ', v.scale)
print('CONFIDENCE', v.confidence)

**Nobody told it to split by app version.** The prompt said "split it, by
whatever column exists", and it worked out which column mattered by looking.

---

## What you learned

- A **tool** is a function the model may call. Its docstring is the interface
- An **agent** is a model, tools, and a loop. `create_agent` is that loop
- **Structured output** turns an answer into fields another program can branch on
- **The prompt is a request. The read only connection is a constraint.** Build on
  constraints
- Give an agent **a method, not a list of known problems**